# Prompt Templates and LCEL Chains

**Course author:** Jayneel Girnara

Move from a one-off prompt to a reusable prompt template and a composable LangChain Expression Language (LCEL) chain.

## Learning goals
- Send role-based messages to a chat model.
- Define a `ChatPromptTemplate` with named input variables.
- Compose prompt, model, and output parser with `|` and invoke the chain with a dictionary.

## Before you start
Install the project dependencies and configure `OPENAI_API_KEY` in a local .env file. Do not place credentials in notebook cells or commit them.

## How to work through it
Run cells in order. First compare a direct model call with a parameterized prompt, then build and invoke the complete chain. Model calls may incur API charges.

## 1. Start with a direct chat prompt

A direct prompt is convenient for a single request, but values and instructions are embedded in that request. A list of role/content messages lets the model distinguish behavior instructions from the user’s task. `.invoke` waits for one completed response.

**Checkpoint:** Which message describes the assistant’s role, and which contains the question?

In [1]:
from langchain_openai import ChatOpenAI

In [2]:
llm=ChatOpenAI(model="gpt-4o")

In [4]:
prompts=[
    {"role":"system","content":"You are a Javascript developer"},
    {"role":"user","content":"Difference b/w map and filter"},
] 

In [6]:
res=llm.invoke(prompts)
print(res.content)

In JavaScript, both `map` and `filter` are array methods that allow you to manipulate elements within an array, but they serve different purposes and have different behaviors:

### `map` Method:
- **Purpose:** The `map` method is used to transform each element in an array and create a new array from the results.
- **Operation:** It applies a given function to each element of the array and constructs a new array from the returned values of this function.
- **Return Value:** Returns a new array of the same length as the original, with the transformed values.
- **Modification:** Does not modify the original array; instead, it produces a new array.

**Example:**
```javascript
const numbers = [1, 2, 3, 4];
const doubled = numbers.map(num => num * 2);
console.log(doubled); // Output: [2, 4, 6, 8]
```

### `filter` Method:
- **Purpose:** The `filter` method is used to select a subset of elements from an array based on specified conditions.
- **Operation:** It applies a predicate function to e

## 2. Parameterize the prompt and build an LCEL chain

A **prompt template** contains named variables such as `{language}` and `{query}`. `format_messages` fills those variables before a model call. LCEL composes compatible steps with `|`: the output of one step becomes the next step’s input. `StrOutputParser` turns an AI message into plain text. Calling `.invoke({...})` supplies the template variables as a dictionary.

**Checkpoint:** What keys must the dictionary passed to `chains.invoke` contain?

In [22]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
 

In [23]:
llm=ChatOpenAI(model="gpt-4o")
out=StrOutputParser()

In [10]:
prompts=ChatPromptTemplate.from_messages([
    {"role":"system","content":"You are a translator and translate input in {language}"},
    {"role":"user","content":"{query}"},
])

prompts

ChatPromptTemplate(input_variables=['language', 'query'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='You are a translator and translate input in {language}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='{query}'), additional_kwargs={})])

In [14]:
final_prompts=prompts.format_messages(language="Hinglish",query="I love python and Javascript")

res=llm.invoke(final_prompts)
print(res.content)

Main Python aur Javascript ko pasand karta hoon.


In [ ]:
# res=llm.invoke(prompts.invoke({"language":"Hindi","query":"I love you"}))
# print(res.content)



मैं तुमसे प्यार करता हूँ / मैं तुमसे प्यार करती हूँ।


In [ ]:
chains = prompts | llm | out

In [18]:
chains

ChatPromptTemplate(input_variables=['language', 'query'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='You are a translator and translate input in {language}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='{query}'), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x113791d60>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x113791f40>, root_client=<openai.OpenAI object at 0x113791dc0>, root_async_client=<openai.AsyncOpenAI object at 0x1137917c0>, model_name='gpt-4o', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_proxy=None, stream_usage=True)

In [25]:
res=chains.invoke({"language":"Hinglish ","query":"I love Langchain"})
print(res.content)

Main Langchain ko pasand karta hoon.
